## 6 - Loading Processed Data

In this step, we load processed video dataset from the previously notebook


In [ ]:
import os
import pandas as pd
import random
import shutil
from tqdm import tqdm

In [ ]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/fei_images.csv"
full_fei_processed_df = pd.read_csv(load_path)

print(f"{len(full_fei_processed_df)} images loaded!")
pd.set_option('display.max_colwidth', None)
display(full_fei_processed_df.sample(15))

## 7 - Data Split (70/15/15) & Identity Isolation

To ensure the scientific validity of the training process, the **FEI Morph** dataset is split using a strict **Identity-Based** approach. Standard random splitting is avoided to prevent **Data Leakage**, which occurs when the model recognizes a specific person's features rather than the actual morphing artifacts.

### The Identity Isolation Strategy
A "Fake" (Morph) image is a combination of two identities: **Subject 1** and **Subject 2**. To guarantee that the model is tested on entirely unseen faces, we implement a **Link-Breaking** strategy:

* **Subject Partitioning**: All 200 unique subjects are randomly assigned to three isolated pools: **Train (70%)**, **Validation (15%)**, and **Test (15%)**.
* **Strict Assignment**: 
    * **Originals**: Assigned based on the subject's pool.
    * **Morphs**: Kept only if **both** involved subjects belong to the same pool.
* **Link Breaking**: Morphs that bridge two different splits (e.g., Subject A in Train and Subject B in Test) are **discarded**.

This rigorous method ensures **zero identity overlap** between splits. While this results in discarding a significant portion of the morphing images (those connecting different pools), it is a necessary requirement to build a model that generalizes to new faces and provides reliable, honest performance metrics.


In [ ]:
def split_fei_by_breaking_links(df, train_size=0.7, val_size=0.15):
    all_subjects = sorted(list(set(df['subj1'].unique())))
    random.seed(42)
    random.shuffle(all_subjects)

    n = len(all_subjects)
    tr_idx = int(n * train_size)
    vl_idx = int(n * (train_size + val_size))

    train_pool = set(all_subjects[:tr_idx])
    val_pool = set(all_subjects[tr_idx:vl_idx])
    test_pool = set(all_subjects[vl_idx:])

    def assign(row):
        s1, s2 = row['subj1'], row['subj2']
        
        if pd.isna(s2) or s2 == "":
            if s1 in train_pool: return 'train'
            if s1 in val_pool: return 'val'
            return 'test'
        
        # Morph images (Fake) - BOTH subjects must be in the same pool
        # This is the "Link Breaking" part
        if s1 in train_pool and s2 in train_pool: return 'train'
        if s1 in val_pool and s2 in val_pool: return 'val'
        if s1 in test_pool and s2 in test_pool: return 'test'
        
        # If Subj1 is in Train but Subj2 is in Test, we DISCARD to prevent leakage
        return 'discard'

    df['split'] = df.apply(assign, axis=1)


    df_final = df[df['split'] != 'discard'].copy()
    
    print(f"--- SPLIT REPORT ---")
    print(f"Total images before: {len(df)}")
    print(f"Total images kept:   {len(df_final)}")
    print(f"Images discarded to prevent leakage: {len(df) - len(df_final)}")
    return df_final

full_fei_processed_df = split_fei_by_breaking_links(full_fei_processed_df)

print("\n--- STATISTICS ---")
print(full_fei_processed_df['split'].value_counts())
full_fei_processed_df.head()

In [ ]:
train_subjs = set(full_fei_processed_df[full_fei_processed_df['split'] == 'train']['subj1'].unique())
val_subjs   = set(full_fei_processed_df[full_fei_processed_df['split'] == 'val']['subj1'].unique())
test_subjs  = set(full_fei_processed_df[full_fei_processed_df['split'] == 'test']['subj1'].unique())

train_test_int = train_subjs.intersection(test_subjs)
train_val_int  = train_subjs.intersection(val_subjs)
val_test_int   = val_subjs.intersection(test_subjs)

print("--- IDENTITY LEAKAGE CHECK ---")

if len(train_test_int) == 0 and len(train_val_int) == 0 and len(val_test_int) == 0:
    print("TEST PASSED: No subjects from the Training set are present in the Validation or Test sets.")
    print(f"Total Unique Subjects: {len(train_subjs | val_subjs | test_subjs)}")
    print(f"  -> Train Subjects: {len(train_subjs)}")
    print(f"  -> Val Subjects:   {len(val_subjs)}")
    print(f"  -> Test Subjects:  {len(test_subjs)}")
else:
    print(f"WARNING: Identity Leakage detected!")
    if len(train_test_int) > 0:
        print(f"  -> {len(train_test_int)} subjects shared between Train and Test.")
    if len(train_val_int) > 0:
        print(f"  -> {len(train_val_int)} subjects shared between Train and Val.")
    if len(val_test_int) > 0:
        print(f"  -> {len(val_test_int)} subjects shared between Val and Test.")

In [ ]:
# CONFIGURATION
source_root = "FEI_Processed" 
output_root = "FEI_MORPHV2_DATASET" 

splits = ['train', 'val', 'test']
categories = ['original', 'fake']

# Create the directory structure
for s in splits:
    for cat in categories:
        os.makedirs(os.path.join(output_root, s, cat), exist_ok=True)

print(f"--- STARTING PHYSICAL FILE COPY TO {output_root} ---")

copy_count = 0
error_count = 0

for idx, row in tqdm(full_fei_processed_df.iterrows(), total=len(full_fei_processed_df)):
    current_path = row['path']
    split_folder = row['split'] # 'train', 'val', or 'test'
    
    # Determine the category folder based on the label
    cat_folder = 'original' if str(row['label']) in ['0', '0.0', 'original'] else 'fake'
    
    new_filename = row['filename']
    destination_path = os.path.join(output_root, split_folder, cat_folder, new_filename)
    
    try:
        if os.path.exists(current_path):
            shutil.copy2(current_path, destination_path)
            # Update path in the dataframe to the new location
            full_fei_processed_df.at[idx, 'path'] = os.path.abspath(destination_path)
            copy_count += 1
        else:
            error_count += 1
    except Exception as e:
        print(f"Error copying {current_path}: {e}")
        error_count += 1

print(f"\n--- COPY COMPLETED ---")
print(f"Files successfully copied: {copy_count}")
print(f"Files missing or errors:   {error_count}")

# Verify file counts per folder
for s in splits:
    for cat in categories:
        p = os.path.join(output_root, s, cat)
        num_files = len([f for f in os.listdir(p) if not f.startswith('.')])
        print(f"Folder {s}/{cat}: {num_files} images")

# Create a clean version containing only the requested columns
full_fei_processed_df['dataset'] = 'FEI'

df_fei_final = full_fei_processed_df[
    ['path', 'label', 'split', 'dataset', 'algorithm', 'subj1', 'subj2']
].copy()

df_fei_final = df_fei_final.rename(columns={
    'algorithm': 'method',
    'subj1': 'target',
    'subj2': 'source'
})

df_fei_final['source'] = df_fei_final['source'].fillna(0).astype(int)
df_fei_final['target'] = df_fei_final['target'].fillna(0).astype(int)

print("\n--- NEW DATAFRAME PREPARED ---")
pd.set_option('display.max_colwidth', None)
print(df_fei_final.sample(10))

## 8 - Sanity Check & Detection Analysis

In [ ]:
print("--- FEI MORPH SANITY CHECK & DETECTION ANALYSIS ---")

def count_methods(base_dir):
    """Counts files on disk, distinguishing between _ssd and _cc suffixes"""
    if not os.path.exists(base_dir):
        return 0, 0, 0
    
    total, ssd, cc = 0, 0, 0
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                total += 1
                if '_ssd' in file:
                    ssd += 1
                elif '_cc' in file:
                    cc += 1
    return total, ssd, cc

# Compare against the SPLIT output (output_root), not the pre-split pool
# (FEI_Processed): df_fei_final only contains the images that survived the
# identity-based train/val/test split, so it must be checked against
# output_root, which is where those surviving images were actually copied.
expected_total = len(df_fei_final)
processed_path = output_root

actual_total, total_ssd, total_cc = count_methods(processed_path)

print(f"\nExpected images (from DataFrame): {expected_total}")
print(f"Found images (on Disk):          {actual_total}")

# Integrity Check
if expected_total == actual_total and expected_total > 0:
    print("STATUS: PERFECT! The number of files matches the DataFrame.")
elif actual_total == 0:
    print("STATUS: ERROR! No files found in the directory.")
else:
    print(f"STATUS: MISMATCH! {expected_total - actual_total} images are missing on disk.")

# Detection Quality Analysis
if actual_total > 0:
    ssd_pct = (total_ssd / actual_total) * 100
    cc_pct = (total_cc / actual_total) * 100
    
    print("\n--- DETECTION QUALITY STATISTICS ---")
    print(f"SSD Face Detector:    {total_ssd} ({ssd_pct:.1f}%)")
    print(f"Center Crop Fallback: {total_cc} ({cc_pct:.1f}%)")
    
print("\n--- CLASS DISTRIBUTION ON DISK ---")
for split_name in splits:
    for label_folder in categories:
        label_path = os.path.join(processed_path, split_name, label_folder)
        if os.path.exists(label_path):
            count = len([f for f in os.listdir(label_path) if os.path.isfile(os.path.join(label_path, f)) and not f.startswith('.')])
            print(f"  -> Folder '{split_name}/{label_folder}': {count} images")

## 9 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_images/fei_images_final.csv`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [ ]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "fei_images_final.csv")
df_fei_final.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")